# Tulya Experiment 2 v3 — Shared-Regime Zero-Shot Prognosis

v2 is closed as a failed preflight. This is a **new experiment**, not a repair of v2.

v3 uses only the three regimes that were empirically shared across heterogeneous domains:

- `no_event`
- `stagnation`
- `overfit_or_memorization`

The v2 seed-0 development runs are excluded. v3 uses fresh seeds **100–105**.

Total corpus: **60 runs** (10 frozen domain/recipe cells × 6 seeds).

There is no further recipe-tuning round. If the v3 corpus validity gate fails, v3 stops.

Enable **GPU T4 ×2** and **Internet**.

In [ ]:
import os, sys, subprocess, importlib, json, time

REPO="/kaggle/working/tulya-training-dynamics"
if os.path.exists(os.path.join(REPO,".git")):
    subprocess.run(["git","-C",REPO,"fetch","origin","main"],check=True)
    subprocess.run(["git","-C",REPO,"reset","--hard","origin/main"],check=True)
else:
    subprocess.run(["git","clone","https://github.com/Vedsaga/tulya-training-dynamics.git",REPO],check=True)

os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0,REPO)

for m in ["experiment2_eval","experiment2_v3","experiment2_core","kaggle_grokking_experiment"]:
    sys.modules.pop(m,None)
importlib.invalidate_caches()

import torch, pandas as pd
commit=subprocess.check_output(["git","-C",REPO,"rev-parse","HEAD"],text=True).strip()
print("commit:",commit)
print("CUDA:",torch.cuda.is_available())
print("visible GPUs:",torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}:",torch.cuda.get_device_name(i))
if torch.cuda.device_count()<2:
    print("WARNING: v3 still runs, but intended speedup requires 2 GPUs.")

## 1. Frozen v3 preregistration

In [ ]:
print(open("EXPERIMENT2_V3_PREREGISTRATION.md").read())

## 2. Run the fresh 60-run corpus

This is the scientific corpus.

There is **no seed-0 preflight** in v3, because seed 0 was already used to design the recipe set. We run fresh untouched seeds 100–105 directly.

The runner is resumable. Completed v3 runs are reused if Kaggle disconnects.

In [ ]:
import importlib
import experiment2_v3
experiment2_v3=importlib.reload(experiment2_v3)

ROOT="/kaggle/working/tulya_exp2_v3"
start=time.time()
manifest=experiment2_v3.run_v3_suite(ROOT)
print("v3 suite wall time:",time.time()-start)

display(manifest.groupby(
    ["domain","recipe_name","recipe_expected_event","event","event_observed"]
).size().rename("runs").reset_index())

## 3. Corpus validity gate

The full 60-run corpus must satisfy the frozen v3 validity rules before any A/B/C/D evaluation.

If this fails, stop. There is no recipe repair inside v3.

In [ ]:
problems,recipe_stability=experiment2_v3.validate_v3_manifest(manifest)

print("Recipe stability:")
display(recipe_stability)

print("\nObserved outcomes by domain:")
display(manifest.groupby(["domain","event"]).size().rename("runs").reset_index())

if problems:
    print("\nV3 CORPUS: INVALID_DATA_GENERATION")
    for p in problems:
        print(" -",p)
    v3_valid=False
else:
    print("\nV3 CORPUS: VALID — blind evaluation allowed.")
    v3_valid=True

## 4. Blind zero-shot evaluation

Core verdict still uses only A/B/C:

- A — learning curves
- B — raw telemetry
- C — normalized/canonical telemetry

D = raw + normalized is diagnostic only and cannot rescue C.

The evaluator also reports bootstrap confidence intervals, within-domain grouped CV, domain identifiability, and false-stop compute utility.

In [ ]:
if not v3_valid:
    raise RuntimeError("v3 corpus invalid; blind evaluator intentionally blocked.")

import experiment2_eval
experiment2_eval=importlib.reload(experiment2_eval)

forecast_table=experiment2_eval.build_table(ROOT)
folds,result,within_domain,domain_identity=experiment2_eval.evaluate(ROOT)

print("\nZERO-SHOT LEAVE-ONE-DOMAIN-OUT:")
display(folds)

print("\nWITHIN-DOMAIN GROUPED DIAGNOSTIC:")
display(within_domain)

print("\nDOMAIN-IDENTITY DIAGNOSTIC:")
display(domain_identity)

print("\nDIAGNOSTIC INTERPRETATION:")
print(json.dumps(result.get("diagnostics",{}),indent=2))

print("\nCORE RESULT:")
print(json.dumps(result,indent=2))

## 5. Decision

Interpret the evaluator's core result using the v3 preregistration:

- core C-vs-B gates pass → **CONTINUE_TO_REAL_LM_EXPERIMENT**
- any core gate fails → **KILL_CANONICAL_TRAINING_DYNAMICS_THESIS**

Do not add new features, learned canonicalization, new thresholds, or new event classes after seeing this result.

In [ ]:
if result["verdict"]=="CONTINUE_TO_EXPERIMENT_3":
    v3_verdict="CONTINUE_TO_REAL_LM_EXPERIMENT"
else:
    v3_verdict="KILL_CANONICAL_TRAINING_DYNAMICS_THESIS"

print("V3 VERDICT:",v3_verdict)
print("\nArtifacts:")
for name in [
    "manifest.csv",
    "forecast_table.csv",
    "evaluation_folds.csv",
    "evaluation_summary.json",
    "diagnostic_in_domain.csv",
    "diagnostic_domain_identity.csv",
    "diagnostic_interpretation.json",
]:
    print(os.path.join(ROOT,name))